In [ ]:
# Notebook 02 – Logistic Regression & Random Forest
# Purpose: Train and evaluate two baseline classifiers on credit card fraud dataset
#          using SMOTE to handle class imbalance and analyze their performance.

# === Import Libraries ===
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    f1_score
)

from imblearn.over_sampling import SMOTE

# === Load and Preprocess Dataset ===
df = pd.read_csv('../data/raw/creditcard.csv')

# Scale 'Amount' and 'Time' columns using StandardScaler
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_scaled'] = StandardScaler().fit_transform(df[['Time']])

# Drop original unscaled columns
df.drop(columns=['Amount', 'Time'], inplace=True)

# Separate features and target label
X = df.drop(columns='Class')
y = df['Class']

# Split data into training and test sets with stratified sampling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# === Apply SMOTE to Balance Training Data ===
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# === Train Logistic Regression Model ===
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_sm, y_train_sm)
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

# === Train Random Forest Model ===
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced')
rf.fit(X_train_sm, y_train_sm)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# === Model Evaluation Utility Function ===
def evaluate_model(y_true, y_pred, y_proba, name):
    print(f"\n{name} Classification Report:")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print(f"ROC-AUC: {roc_auc_score(y_true, y_proba):.4f}")
    f1 = f1_score(y_true, y_pred)
    print(f"F1-score: {f1:.4f}")
    return f1

# === Evaluate Both Models ===
evaluate_model(y_test, y_pred_lr, y_proba_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_rf, y_proba_rf, "Random Forest")

# === Plot Precision-Recall Curve for Random Forest ===
precision, recall, _ = precision_recall_curve(y_test, y_proba_rf)

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, color='navy')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Random Forest)')
plt.grid(True)
plt.tight_layout()
plt.show()
